In [2]:
#!/usr/bin/env python3
# =============================================================================
# regenerate_figures.py
# Regenerate Figure 7 (ablation) and Figure 6 (hallucination) from the
# primary-policy honest-evaluation results.
#
# Reads:
#   ../results/tables/final_eval_summary.csv    (from 05d)
#   ../results/tables/final_eval_percf.csv      (from 05d)
#   ../results/tables/final_eval_feasibility.csv(from 05d)
#
# Writes (600 dpi PNG/PDF/TIFF, matching the paper's figure convention):
#   ../results/figures/figure_ablation.{png,pdf,tiff}
#   ../results/figures/figure_hallucination.{png,pdf,tiff}
#
# Run from the notebooks/ directory (paths are relative), or adjust BASE.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = os.environ.get("REVISION_BASE", "../results")
TABLES = os.path.join(BASE, "tables")
FIGS = os.path.join(BASE, "figures")
os.makedirs(FIGS, exist_ok=True)

# Paper palette (blue = pure DiCE, orange/red = guardrail)
C_PURE = "#4C72B0"
C_GUARD = "#C44E52"
C_GUARD2 = "#DD8452"

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "figure.dpi": 100,
})


def save_all(fig, name):
    for ext in ("png", "pdf", "tiff"):
        fig.savefig(os.path.join(FIGS, f"{name}.{ext}"),
                    dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {name} (png/pdf/tiff @600dpi)")


# -----------------------------------------------------------------------------
# Load results (with a safe fallback so the script is runnable for a dry check)
# -----------------------------------------------------------------------------
def load_summary():
    path = os.path.join(TABLES, "final_eval_summary.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    # Fallback = the primary-policy numbers reported in the paper draft.
    print("  [warn] final_eval_summary.csv not found; using draft fallback.")
    return pd.DataFrame({
        "Policy":   ["primary", "primary", "primary"],
        "Stage":    ["PureDiCE", "soft", "hard"],
        "V_energy_%":   [4.2, 8.5, 0.0],
        "V_conflict_%": [33.3, 0.0, 0.0],
        "V_bmiwt_%":    [8.3, 0.0, 0.0],
        "n_CF":     [48, 47, 47],
    })


def load_feasibility():
    path = os.path.join(TABLES, "final_eval_feasibility.csv")
    if os.path.exists(path):
        fe = pd.read_csv(path)
        return fe
    print("  [warn] final_eval_feasibility.csv not found; using draft fallback.")
    return None


# -----------------------------------------------------------------------------
# Figure 6 (hallucination): three violation types x three conditions
# -----------------------------------------------------------------------------
def figure_hallucination(summary):
    s = summary[summary["Policy"].isin(["primary", "-"])] \
        if "Policy" in summary.columns else summary
    # order stages
    order = ["PureDiCE", "soft", "hard"]
    s = s.set_index("Stage").reindex([o for o in order if o in set(s["Stage"])]) \
        if "Stage" in summary.columns else summary
    # robust extraction
    def row(stage):
        r = summary[(summary["Stage"] == stage)]
        r = r[r["Policy"].isin(["primary", "-"])] if "Policy" in summary.columns else r
        return r.iloc[0]
    pure = row("PureDiCE"); soft = row("soft"); hard = row("hard")

    types = ["Energy floor\n(<800 kcal)",
             "Sodium-carb\nconflict",
             "BMI-weight\ninconsistency"]
    pure_v = [pure["V_energy_%"], pure["V_conflict_%"], pure["V_bmiwt_%"]]
    soft_v = [soft["V_energy_%"], soft["V_conflict_%"], soft["V_bmiwt_%"]]
    hard_v = [hard["V_energy_%"], hard["V_conflict_%"], hard["V_bmiwt_%"]]

    x = np.arange(len(types)); w = 0.26
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    b1 = ax.bar(x - w, pure_v, w, label="Pure DiCE", color=C_PURE)
    b2 = ax.bar(x,      soft_v, w, label="Guardrail (injected)", color=C_GUARD2)
    b3 = ax.bar(x + w,  hard_v, w, label="Guardrail (enforced)", color=C_GUARD)
    for bars in (b1, b2, b3):
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:.1f}", (b.get_x() + b.get_width()/2, h),
                        ha="center", va="bottom", fontsize=7,
                        xytext=(0, 1), textcoords="offset points")
    ax.set_xticks(x); ax.set_xticklabels(types)
    ax.set_ylabel("Violation rate (\\%)".replace("\\%", "%"))
    ax.set_ylim(0, max(pure_v + soft_v + hard_v) * 1.25 + 1)
    ax.set_title("Physiologically implausible recourse (external scoring)")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    save_all(fig, "figure_hallucination")


# -----------------------------------------------------------------------------
# Figure 7 (ablation): 4 panels
#   P1 feasibility, P2 any-violation, P3 violation by type, P4 sparsity
# -----------------------------------------------------------------------------
def figure_ablation(summary, percf, feas):
    def row(stage):
        r = summary[(summary["Stage"] == stage)]
        r = r[r["Policy"].isin(["primary", "-"])] if "Policy" in summary.columns else r
        return r.iloc[0]
    pure = row("PureDiCE")
    guard = row("hard")   # enforced guardrail as the guardrail condition

    fig, axes = plt.subplots(1, 4, figsize=(11, 3.2))

    # Panel 1: feasibility rate (mean CFs/case normalised to target 4, or from feas)
    if feas is not None and "n_CF" in feas.columns:
        pure_feas = feas[(feas["Stage"] == "PureDiCE")]["n_CF"].mean() / 4 * 100
        gd = feas[(feas["Stage"] == "soft")]
        gd = gd[gd["Policy"] == "primary"] if "Policy" in feas.columns else gd
        guard_feas = gd["n_CF"].mean() / 4 * 100
    else:
        pure_feas, guard_feas = 100.0, 98.0
    axes[0].bar(["Pure\nDiCE", "Guardrail"], [pure_feas, guard_feas],
                color=[C_PURE, C_GUARD])
    axes[0].set_title("Feasibility rate (%)")
    axes[0].set_ylim(0, 110)

    # Panel 2: any-violation rate among feasible CFs
    def any_viol(stage):
        if percf is None:
            return None
        sub = percf[percf["Stage"] == stage] if "Stage" in percf.columns else \
              percf[percf["Condition"] == stage]
        if "Policy" in sub.columns:
            sub = sub[sub["Policy"].isin(["primary", "-"])]
        if len(sub) == 0:
            return None
        v = ((sub["V_energy"] + sub["V_conflict"] + sub["V_bmiwt"]) > 0).mean() * 100
        return v
    pv = any_viol("PureDiCE"); gv = any_viol("hard")
    if pv is None:
        pv, gv = 45.8, 0.0
    axes[1].bar(["Pure\nDiCE", "Guardrail"], [pv, gv], color=[C_PURE, C_GUARD])
    axes[1].set_title("Any-violation rate (%)")
    axes[1].set_ylim(0, max(pv, gv) * 1.25 + 1)

    # Panel 3: violation breakdown by type
    types = ["Anthro.", "Energy", "Conflict"]
    pure_b = [pure["V_bmiwt_%"], pure["V_energy_%"], pure["V_conflict_%"]]
    guard_b = [guard["V_bmiwt_%"], guard["V_energy_%"], guard["V_conflict_%"]]
    x = np.arange(3); w = 0.35
    axes[2].bar(x - w/2, pure_b, w, label="Pure DiCE", color=C_PURE)
    axes[2].bar(x + w/2, guard_b, w, label="Guardrail", color=C_GUARD)
    axes[2].set_xticks(x); axes[2].set_xticklabels(types)
    axes[2].set_title("Violation by type (%)")
    axes[2].legend(frameon=False, fontsize=7)
    axes[2].set_ylim(0, max(pure_b + guard_b) * 1.25 + 1)

    # Panel 4: recourse sparsity (mean changed vars) — from percf if available
    def sparsity(stage):
        if percf is not None and "n_changed" in percf.columns:
            sub = percf[percf["Stage"] == stage] if "Stage" in percf.columns else \
                  percf[percf["Condition"] == stage]
            if "Policy" in sub.columns:
                sub = sub[sub["Policy"].isin(["primary", "-"])]
            if len(sub):
                return sub["n_changed"].mean()
        return None
    ps = sparsity("PureDiCE"); gs = sparsity("hard")
    if ps is None:
        ps, gs = 11.6, 11.0   # draft fallback
    axes[3].bar(["Pure\nDiCE", "Guardrail"], [ps, gs], color=[C_PURE, C_GUARD])
    axes[3].set_title("Avg. changed variables")
    axes[3].set_ylim(0, max(ps, gs) * 1.25 + 1)

    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    save_all(fig, "figure_ablation")


if __name__ == "__main__":
    summary = load_summary()
    feas = load_feasibility()
    percf_path = os.path.join(TABLES, "final_eval_percf.csv")
    percf = pd.read_csv(percf_path) if os.path.exists(percf_path) else None

    print("Regenerating figures from primary-policy results...")
    figure_hallucination(summary)
    figure_ablation(summary, percf, feas)
    print("Done. Note: energy 800 kcal threshold and primary policy per paper.")

Regenerating figures from primary-policy results...
  saved: figure_hallucination (png/pdf/tiff @600dpi)
  saved: figure_ablation (png/pdf/tiff @600dpi)
Done. Note: energy 800 kcal threshold and primary policy per paper.
